# Session 4: RAG Pipeline — Retrieval-Augmented Generation

**Goal:** Build a complete RAG system that answers questions about Croatian history using Wikipedia articles.

| What you'll learn | Why it matters |
|---|---|
| Chunking strategies | Right chunk size = good retrieval |
| Embedding & indexing a real corpus | The offline half of RAG |
| Retrieval + prompt construction | The online half of RAG |
| Grounded generation | LLM answers from evidence, not guesses |

**Dataset:** ~4,100 Croatian Wikipedia articles on history and biography.

**Time:** ~45 minutes

## Setup

In [1]:
!pip install chromadb openai tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the s

In [2]:
import json
import requests
from typing import List, Dict
from openai import OpenAI
from tqdm import tqdm
import chromadb

# ── API endpoint ──
BASE_URL = "https://llmapi.levara.xyz/v1"
API_KEY  = "alumni-llm-workshop-2026-ZAMJENI"  # ← zamijeni s ključem koji dobiješ na radionici

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

CHAT_MODEL  = "cyankiwi/gemma-4-26B-A4B-it-AWQ-8bit"
EMBED_MODEL = "Qwen/Qwen3-Embedding-8B"

# Quick check: embedding dimensions
test_emb = client.embeddings.create(input=["test"], model=EMBED_MODEL)
EMBED_DIM = len(test_emb.data[0].embedding)

print(f"Chat model:      {CHAT_MODEL}")
print(f"Embed model:     {EMBED_MODEL} ({EMBED_DIM} dimensions)")
print(f"✓ Ready")

Chat model:      cyankiwi/gemma-4-26B-A4B-it-AWQ-8bit
Embed model:     Qwen/Qwen3-Embedding-8B (4096 dimensions)
✓ Ready


In [3]:
def chat(messages, temperature=0.7):
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

print("✓ chat() helper defined")

✓ chat() helper defined


## Part 0: The Payoff

Before we build anything, let's see **why** we need RAG.

We'll ask the model a question about Croatian history — **without** any context.

In [4]:
question = "Objasni značaj Bašćanske ploče za hrvatsku kulturu i povijest."

answer_no_rag = chat([
    {"role": "user", "content": question}
])

print(f"Question: {question}\n")
print(f"Answer (no RAG):\n{answer_no_rag}")
print("\n" + "=" * 80)
print("\u2191 Remember this answer. We'll come back at the end with RAG.")

Question: Objasni značaj Bašćanske ploče za hrvatsku kulturu i povijest.

Answer (no RAG):
Bašćanska ploča predstavlja jedan od najvažnijih spomenika hrvatske kulture, jezika i povijesti. Njezina važnost nije samo u materijalnom obliku (kamenu ploči), već u simboličkom značenju koje nosi za identitet hrvatskog naroda.

Evo detaljnog objašnjenja njezina značaja kroz nekoliko ključnih aspekata:

### 1. Povijesni značaj: Dokaz o postojanju države i naroda
Bašćanska ploča datira iz **neprije 1100. godine** (kraj 11. stoljeća). Ona je jedan od najstarijih sačuvanih spomenika u kojem se izravno spominje hrvatski kralj.
*   **Spominjanje kralja:** Na ploči piše da ju je podario kralj **Zvonimir**. To je ključni povijesni dokaz koji potvrđuje postojanje hrvatske državne vlasti i kraljevske titule u tom razdoblju.
*   **Politička legitimacija:** Ploča služi kao materijalni dokaz o političkoj organizaciji i suverenitetu tadašnje Hrvatske.

### 2. Jezikovni i lingvistički značaj: Temelj hrvatskog

## Part 1: Load the Corpus

Our dataset: **~4,100 Croatian Wikipedia articles** about history and biography.

Each article has structured sections extracted from Wikipedia — we'll use this structure for chunking.

In [7]:
# Upload hrwiki_history_bio.jsonl using the Files panel (📁) on the left,
# or run:  from google.colab import files; uploaded = files.upload()

def load_corpus(path="hrwiki_history_bio.jsonl"):
    """Load articles from JSONL. Each article has: id, title, sections, categories."""
    articles = []
    with open(path) as f:
        for line in f:
            articles.append(json.loads(line))
    return articles

articles = load_corpus()

print(f"\u2713 Loaded {len(articles)} articles")
print(f"\nSample titles:")
for a in articles[:8]:
    n_sections = len(a['sections'])
    print(f"  {a['title']} ({a['char_count']} chars, {n_sections} sections)")
print(f"  ...")

✓ Loaded 4104 articles

Sample titles:
  Pacta conventa (5935 chars, 2 sections)
  Antemurale Christianitatis (4960 chars, 2 sections)
  Zrinsko-frankopanska urota (13635 chars, 11 sections)
  Hrvatska u Revoluciji 1848. (15224 chars, 10 sections)
  Bašćanska ploča (7751 chars, 9 sections)
  Petar Berislavić (3818 chars, 4 sections)
  Petar Kružić (10570 chars, 8 sections)
  Nikola VII. Zrinski (25458 chars, 18 sections)
  ...


In [8]:
# Look at one article's structure
article = next(a for a in articles if a["title"] == "Ba\u0161\u0107anska plo\u010da")

print(f"Title: {article['title']}")
print(f"Total chars: {article['char_count']}")
print(f"Categories: {', '.join(article['categories'][:5])}")
print(f"\nSections ({len(article['sections'])}):\n")
for s in article["sections"]:
    print(f"  [{len(s['text']):>5} chars]  {s['heading']}")
    print(f"             {s['text'][:80]}...\n")

Title: Bašćanska ploča
Total chars: 7751
Categories: Arheološki artefakti u Hrvatskoj, Povijest hrvatskoga jezika, Hrvatska za narodnih vladara, Spomenici u Hrvatskoj, Baška

Sections (9):

  [ 1098 chars]  Uvod
             mini|250px|Bašćanska ploča izložena u HAZU. mini|250px|Crkva sv. Lucije na Krku....

  [  571 chars]  Izgled
             Izgled Ploča je izvorno bila lijevi plutej oltarne pregrade koja je dijelila pre...

  [ 1559 chars]  Otkriće
             Otkriće Ploča je do sredine 18. stoljeća stajala kao lijevi plutej oltarne pregr...

  [  428 chars]  Datiranje
             Datiranje Na početku teksta pojedini vide invokaciju (+), drugi invokaciju i slo...

  [ 1182 chars]  Sadržaj
             Sadržaj Tekst ploče ima sljedeći sadržaj: invokacija (zazivanje Boga) zapis opat...

  [  274 chars]  Pismo
             Pismo Pisana je prijelaznim tipom glagoljice, s oble na uglatu. Usporedno s glag...

  [ 1955 chars]  Prijepis
             Prijepis Amaterski prijepis teksta pl

## Part 2: Chunking Strategies

We can't embed entire articles as single vectors — a 10-page article covers many topics.
One vector would be an **average of everything**, matching everything vaguely, nothing precisely.

**Chunking** splits documents into smaller pieces, each capturing a focused unit of meaning.

| Too small | Too large |
|---|---|
| Loses context | Dilutes relevance |
| "It is 100 degrees" — temperature? angle? | 3 pages of text, 1 sentence answers the question |

### Strategy 1: Fixed-Size Chunking

Split every N characters. Simple, but cuts mid-sentence, mid-paragraph, mid-thought.

In [9]:
def chunk_fixed(text, size=500, overlap=50):
    """Naive fixed-size chunking at character boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])
        start += size - overlap
    return chunks


# Demo: fixed-size chunks on Ba\u0161\u0107anska plo\u010da
full_text = "\n\n".join(s["text"] for s in article["sections"])
fixed_chunks = chunk_fixed(full_text, size=500, overlap=50)

print(f"Article: {article['title']} ({len(full_text)} chars)")
print(f"Fixed-size chunks: {len(fixed_chunks)} (size=500, overlap=50)\n")

for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)
    print()

Article: Bašćanska ploča (7767 chars)
Fixed-size chunks: 18 (size=500, overlap=50)

--- Chunk 0 (500 chars) ---
mini|250px|Bašćanska ploča izložena u HAZU. mini|250px|Crkva sv. Lucije na Krku. mini|250px|Unutrašnjost crkve sv. Lucije. mini|upright=1.35|tekst ploče na starohrvatskom jeziku Bašćanska ploča starohrvatski je spomenik pisan prijelaznim oblikom glagoljice oko 1100. godine. Na njoj je dokumentirano kako je hrvatski kralj Dmitar Zvonimir mjesnom benediktinskom samostanu darovao zemlju. Pronađena je 15. rujna 1851. godine u crkvi sv. Lucije u Jurandvoru kod Baške na otoku Krku zahvaljujući tadašnj

--- Chunk 1 (500 chars) ---
dvoru kod Baške na otoku Krku zahvaljujući tadašnjemu bogoslovu Petru Dorčiću. Predstavlja značajan izvor za povijest hrvatskog naroda, jezika i razvitak hrvatske glagoljice. Pokazuje suverenitet hrvatskoga kralja Zvonimira kao donatora zemljišnog posjeda na otoku. Uz jezično i književno, ta ploča ima i povijesno značenje jer se prvi put na hrvatskom jezik

Notice how chunks break in the middle of sentences. The model retrieves a fragment that starts and ends mid-thought.

### Strategy 2: Section-Based Chunking

Our Wikipedia data already has **sections with headings**. We can respect this structure:
- Small sections → keep as one chunk
- Large sections → split at sentence boundaries with overlap

In [10]:
# Sections to skip — boilerplate that adds noise, not knowledge
SKIP_SECTIONS = {
    "Vanjske poveznice", "Izvori", "Literatura", "Vidi još",
    "Povezani članci", "Bilješke", "Poveznice", "Izvor",
    "Sinkronizacija", "Filmografija", "Diskografija",
    "Filmske uloge", "Televizijske uloge", "Mrežna sjedišta",
    "Ostalo", "Galerija", "Napomene",
}

MIN_CHUNK_CHARS = 200  # skip tiny fragments


def chunk_by_sections(article, max_chunk=1500, overlap=100):
    """Section-aware chunking. Skips boilerplate sections,
    splits large sections at sentence boundaries."""
    chunks = []
    title = article["title"]

    for section in article["sections"]:
        heading = section["heading"]
        text = section["text"]

        # Skip boilerplate and tiny sections
        if heading in SKIP_SECTIONS:
            continue
        if len(text) < MIN_CHUNK_CHARS:
            continue

        if len(text) <= max_chunk:
            chunks.append({
                "text": text,
                "title": title,
                "section": heading,
            })
        else:
            # Split large sections at sentence boundaries
            start = 0
            while start < len(text):
                end = min(start + max_chunk, len(text))
                # Try to break at a sentence boundary
                if end < len(text):
                    for sep in [". ", "? ", "! "]:
                        pos = text.rfind(sep, start + max_chunk // 2, end)
                        if pos != -1:
                            end = pos + len(sep)
                            break
                chunk = text[start:end].strip()
                if chunk and len(chunk) >= MIN_CHUNK_CHARS:
                    chunks.append({
                        "text": chunk,
                        "title": title,
                        "section": heading,
                    })
                # Always advance by at least (max_chunk - overlap)
                new_start = end - overlap
                start = max(new_start, start + max_chunk - overlap)

    return chunks


# Demo: section-based chunks on the same article
section_chunks = chunk_by_sections(article)

print(f"Section-based chunks: {len(section_chunks)} (max=1500, overlap=100)\n")
print(f"Skipped sections: {', '.join(SKIP_SECTIONS)}\n")

for i, chunk in enumerate(section_chunks[:4]):
    print(f"--- Chunk {i}: [{chunk['section']}] ({len(chunk['text'])} chars) ---")
    print(chunk["text"][:200])
    print("...\n")

Section-based chunks: 9 (max=1500, overlap=100)

Skipped sections: Filmske uloge, Vidi još, Sinkronizacija, Diskografija, Izvor, Literatura, Ostalo, Poveznice, Mrežna sjedišta, Izvori, Filmografija, Povezani članci, Napomene, Bilješke, Televizijske uloge, Vanjske poveznice, Galerija

--- Chunk 0: [Uvod] (1098 chars) ---
mini|250px|Bašćanska ploča izložena u HAZU. mini|250px|Crkva sv. Lucije na Krku. mini|250px|Unutrašnjost crkve sv. Lucije. mini|upright=1.35|tekst ploče na starohrvatskom jeziku Bašćanska ploča staroh
...

--- Chunk 1: [Izgled] (571 chars) ---
Izgled Ploča je izvorno bila lijevi plutej oltarne pregrade koja je dijelila prezbiterij (prostor za svećenstvo) od crkvene lađe (prostor za puk). Jurandvorski ulomci činili bi desni plutej crkve. Plo
...

--- Chunk 2: [Otkriće] (1478 chars) ---
Otkriće Ploča je do sredine 18. stoljeća stajala kao lijevi plutej oltarne pregrade u crkvi sv. Lucije. Kasnije je postavljena na pod kao nadgrobna ploča. Na nju je prvi put 1851. upozorio

In [ ]:
# Side-by-side comparison
print(f"Article: {article['title']} ({len(full_text)} chars)\n")
print(f"Fixed-size (500 chars):     {len(fixed_chunks):>3} chunks, avg {sum(len(c) for c in fixed_chunks) // len(fixed_chunks)} chars")
print(f"Section-based (1500 chars): {len(section_chunks):>3} chunks, avg {sum(len(c['text']) for c in section_chunks) // len(section_chunks)} chars")
print(f"\n→ Section-based chunks preserve headings and sentence boundaries.")
print(f"→ Boilerplate sections (links, references) are skipped.")
print(f"→ We'll use section-based chunking for the rest of this notebook.")

## Part 3: Embed & Store in ChromaDB

Now we chunk the **entire corpus** and store it in ChromaDB.

Pipeline: **Articles → Chunk → Embed → Store**

ChromaDB will auto-embed each chunk using our embedding function.

In [11]:
# ── Embedding function for ChromaDB ──
class VLLMEmbeddingFunction:
    """Bridges ChromaDB with our vLLM embedding endpoint."""

    def __init__(self, client, model):
        self.client = client
        self.model = model

    def __call__(self, input: List[str]) -> List[List[float]]:
        response = self.client.embeddings.create(input=input, model=self.model)
        return [item.embedding for item in response.data]

    def embed_query(self, input) -> List[List[float]]:
        texts = input if isinstance(input, list) else [input]
        return self(texts)

    def name(self) -> str:
        return self.model


embedding_fn = VLLMEmbeddingFunction(client, EMBED_MODEL)

# ── Create ChromaDB collection ──
db = chromadb.Client()
collection = db.get_or_create_collection(
    name="hrwiki_history",
    embedding_function=embedding_fn,
)

print(f"✓ Collection '{collection.name}' ready")
print(f"  Embedding model: {EMBED_MODEL} ({EMBED_DIM}d)")

✓ Collection 'hrwiki_history' ready
  Embedding model: Qwen/Qwen3-Embedding-8B (4096d)


In [13]:
# ── Chunk and index in streaming batches ──
# Process articles one by one, flush to ChromaDB every BATCH_SIZE chunks.
# This avoids holding all chunks in memory at once.

BATCH_SIZE = 128

batch_docs, batch_ids, batch_metas = [], [], []
total_chunks = 0

for article in tqdm(articles, desc="Chunking & indexing"):
    chunks = chunk_by_sections(article)
    for i, chunk in enumerate(chunks):
        batch_docs.append(chunk["text"])
        batch_ids.append(f"{article['id']}_{i}")
        batch_metas.append({"title": chunk["title"], "section": chunk["section"]})

        if len(batch_docs) >= BATCH_SIZE:
            collection.add(documents=batch_docs, ids=batch_ids, metadatas=batch_metas)
            total_chunks += len(batch_docs)
            batch_docs, batch_ids, batch_metas = [], [], []

# Flush remaining
if batch_docs:
    collection.add(documents=batch_docs, ids=batch_ids, metadatas=batch_metas)
    total_chunks += len(batch_docs)

print(f"\n✓ Indexed {total_chunks} chunks from {len(articles)} articles")

Chunking & indexing: 100%|██████████| 4104/4104 [1:16:46<00:00,  1.12s/it]



✓ Indexed 21794 chunks from 4104 articles


## Part 4: The RAG Pipeline

Now we connect **retrieval** (ChromaDB) with **generation** (LLM).

For every question:
1. **Retrieve** — find the most relevant chunks
2. **Build prompt** — inject retrieved context + the question
3. **Generate** — LLM answers grounded in the context

In [14]:
def retrieve(question: str, top_k: int = 3) -> List[Dict]:
    """Find the most relevant chunks for a question."""
    results = collection.query(query_texts=[question], n_results=top_k)
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({
            "text": doc,
            "title": meta["title"],
            "section": meta["section"],
            "similarity": 1 - dist,
        })
    return chunks


# Test retrieval
results = retrieve("Ba\u0161\u0107anska plo\u010da", top_k=3)
print("Retrieval test: 'Ba\u0161\u0107anska plo\u010da'\n")
for r in results:
    print(f"  [{r['similarity']:.3f}] {r['title']} \u2014 {r['section']}")
    print(f"          {r['text'][:100]}...\n")

Retrieval test: 'Bašćanska ploča'

  [0.350] Bašćanska ploča — Uvod
          mini|250px|Bašćanska ploča izložena u HAZU. mini|250px|Crkva sv. Lucije na Krku. mini|250px|Unutrašn...

  [0.232] Bašćanska ploča — Sadržaj
          Sadržaj Tekst ploče ima sljedeći sadržaj: invokacija (zazivanje Boga) zapis opata Držihe u prvom lic...

  [0.192] Bašćanska ploča — Prijepis
          Prijepis Amaterski prijepis teksta ploče Ivan Kukuljević Sakcinski poslao je Pavelu Šafáriku (1853.)...



### The RAG Prompt

The key instruction: **answer ONLY from the provided context.**

Without this, the LLM falls back to its training data and may hallucinate.
With it, the LLM is grounded — every claim should trace back to a retrieved chunk.

In [15]:
SYSTEM_PROMPT = """You are a helpful assistant answering questions about Croatian history and culture.

RULES:
1. Answer ONLY based on the provided context.
2. If the context doesn't contain enough information, say: "Nemam dovoljno informacija u zadanom kontekstu."
3. Do NOT use knowledge from your training data — only the provided context.
4. Answer in the same language as the question.
5. When possible, mention which article the information comes from."""


def build_prompt(question: str, chunks: List[Dict]) -> List[Dict]:
    """Assemble the RAG prompt: system + context + question."""
    context = "\n\n---\n\n".join(
        f"[Izvor: {c['title']} \u2014 {c['section']}]\n{c['text']}"
        for c in chunks
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Kontekst:\n\n{context}\n\nPitanje: {question}"},
    ]


def rag_query(question: str, top_k: int = 3, temperature: float = 0.3) -> Dict:
    """Full RAG pipeline: retrieve \u2192 build prompt \u2192 generate."""
    chunks = retrieve(question, top_k=top_k)
    messages = build_prompt(question, chunks)
    answer = chat(messages, temperature=temperature)
    return {"answer": answer, "sources": chunks}


print("\u2713 RAG pipeline defined: retrieve() \u2192 build_prompt() \u2192 rag_query()")

✓ RAG pipeline defined: retrieve() → build_prompt() → rag_query()


In [16]:
def show_rag_result(question: str, result: Dict):
    """Pretty-print a RAG result with sources."""
    print(f"Question: {question}")
    print("=" * 80)
    print(f"\n{result['answer']}\n")
    print("-" * 80)
    print("Sources:")
    for s in result["sources"]:
        print(f"  [{s['similarity']:.3f}] {s['title']} \u2014 {s['section']}")

In [21]:
# Test the full pipeline
result = rag_query("Tko je bio Petar Kru\u017ei\u0107 i kako je branio Klis?")
show_rag_result("Tko je bio Petar Kru\u017ei\u0107 i kako je branio Klis?", result)

Question: Tko je bio Petar Kružić i kako je branio Klis?

Petar Kružić je bio hrvatski vojskovođa, kapetan kliški i senjski, te knez kliški (Izvor: Petar Kružić — Uvod). Bio je jedan od najvažnijih hrvatskih ratnika u protuturskim ratovima (Izvor: Petar Kružić — Uvod).

Klis je branio na sljedeći način:
*   Svoju vojnu karijeru počeo je 1513. godine priključivši se braniteljima Klisa, gdje je ispočetka obavljao dužnost podkaštelana, a potom je od 1518. ili 1519. godine postao kapetan Klisa (Izvor: Petar Kružić — Vojno djelovanje).
*   Tijekom opsade Klisa u veljači 1524. godine, kada je turski vojvoda Mustafa opkolio grad s 3000 vojnika, Kružić je u Senju okupio vojsku od 1500 pješaka i 60 konjanika (uz pomoć Toma Nigera koji je donio Papinu pomoć u novcu, naoružanju i hrani). S 40 brodova ploveći noću, stigao je do Solina gdje su se iskrcali 10. travnja, a zatim su krenuli prema Klisu gdje su razbili vojsku koja je opsjedala tvrđavu (Izvor: Petar Kružić — Vojno djelovanje). U tu akcij

In [22]:
result = rag_query("\u0160to je bila Zrinsko-frankopanska urota?")
show_rag_result("\u0160to je bila Zrinsko-frankopanska urota?", result)

Question: Što je bila Zrinsko-frankopanska urota?

Zrinsko-frankopanska urota bio je pokret hrvatskog i ugarskog plemstva protiv apsolutističke politike Habsburgovaca u razdoblju od 1664. do 1671. godine [Izvor: Zrinsko-frankopanska urota — Uvod]. Pokušali su se oduprijeti nastojanjima bečkog dvora pod Leopoldom I. da Hrvatskoj i Ugarskoj nametne centralizam i apsolutizam kakvim se već upravljalo u Austrijskim nasljednim zemljama, uključujući i Češku [Izvor: Zrinsko-frankopanska urota — Uvod]. Urota je za Hrvate i Mađare 1671. godine završila tragično javnim smaknućem Petra Zrinskog i Frana Krsta Frankopana [Izvor: Opsada Novog Zrina — Ogorčenje Nikole Zrinskog i Zrinsko-frankopanska urota].

--------------------------------------------------------------------------------
Sources:
  [0.494] Zrinsko-frankopanska urota — Uvod
  [0.309] Opsada Novog Zrina — Ogorčenje Nikole Zrinskog i Zrinsko-frankopanska urota
  [0.162] Zrinsko-frankopanska urota — Nakon urote


In [23]:
result = rag_query("Kako je Hrvatska u\u0161la u personalnu uniju s Ugarskom?")
show_rag_result("Kako je Hrvatska u\u0161la u personalnu uniju s Ugarskom?", result)

Question: Kako je Hrvatska ušla u personalnu uniju s Ugarskom?

Do stvaranja personalne unije došlo je nakon sukcesijske krize i ratnih događanja krajem 11. i početkom 12. stoljeća u Hrvatskom Kraljevstvu (Izvor: Hrvatsko Kraljevstvo (1102. – 1526.) — Uvod).

--------------------------------------------------------------------------------
Sources:
  [0.434] Hrvatsko Kraljevstvo (1102. – 1526.) — Uvod
  [0.280] Hrvatsko Kraljevstvo — Personalna unija s Ugarskom: kralj Koloman, 1102.
  [0.206] Hrvatska pravna povijest — Državnopravni položaj hrvatskih zemalja u vrijeme personalne unije


## Part 5: The Comparison

Now let's answer the **same question** from Part 0 — this time with RAG.

In [24]:
question = "Objasni zna\u010daj Ba\u0161\u0107anske plo\u010de za hrvatsku kulturu i povijest."

result = rag_query(question)

print("WITHOUT RAG:")
print("=" * 80)
print(answer_no_rag)
print()
print("WITH RAG:")
print("=" * 80)
print(result["answer"])
print()
print("-" * 80)
print("Sources:")
for s in result["sources"]:
    print(f"  [{s['similarity']:.3f}] {s['title']} \u2014 {s['section']}")

WITHOUT RAG:
Bašćanska ploča predstavlja jedan od najvažnijih spomenika hrvatske kulture, jezika i povijesti. Njezina važnost nije samo u materijalnom obliku (kamenu ploči), već u simboličkom značenju koje nosi za identitet hrvatskog naroda.

Evo detaljnog objašnjenja njezina značaja kroz nekoliko ključnih aspekata:

### 1. Povijesni značaj: Dokaz o postojanju države i naroda
Bašćanska ploča datira iz **neprije 1100. godine** (kraj 11. stoljeća). Ona je jedan od najstarijih sačuvanih spomenika u kojem se izravno spominje hrvatski kralj.
*   **Spominjanje kralja:** Na ploči piše da ju je podario kralj **Zvonimir**. To je ključni povijesni dokaz koji potvrđuje postojanje hrvatske državne vlasti i kraljevske titule u tom razdoblju.
*   **Politička legitimacija:** Ploča služi kao materijalni dokaz o političkoj organizaciji i suverenitetu tadašnje Hrvatske.

### 2. Jezikovni i lingvistički značaj: Temelj hrvatskog jezika
Za lingviste, Bašćanska ploča je "sveti gral" hrvatske filologije.
*  

### How many chunks?

Let's see how `top_k` affects the answer. More chunks = more context, but also more noise and cost.

In [25]:
question = "Koja je bila uloga bana Jela\u010di\u0107a u revoluciji 1848.?"

for k in [1, 3, 5, 10]:
    result = rag_query(question, top_k=k)
    # Count approximate tokens (rough: 1 token ~ 4 chars for Croatian)
    context_chars = sum(len(s["text"]) for s in result["sources"])
    print(f"\n--- top_k={k} ({context_chars} chars of context) ---")
    print(result["answer"][:300])
    if len(result["answer"]) > 300:
        print("...")


--- top_k=1 (1392 chars of context) ---
Josip Jelačić je postavljen za bana, a u kontekstu revolucije 1848. godine on je prekinuo sve odnose s Ugarskom, odbacio odluku o zajedničkoj vladi iz 1790., ukinuo kmetstvo i ustrojio samostalnu hrvatsku vladu. Zbog prijetnji iz Mađarske počeo se spremati za obranu zemlje, a s hrvatskom vojskom pre
...

--- top_k=3 (4117 chars of context) ---
Uloga bana Josipa Jelačića u revoluciji 1848. bila je sljedeća:

*   **Postavljanje i funkcije:** Za bana je postavljen pukovnik Josip Jelačić [Izvor: Hrvatska pod Habsburgovcima — Ban Jelačić i sukob s Mađarima (1848.)] i [Izvor: Hrvatska u Revoluciji 1848. — Uvod]. Dana 8. travnja postao je podmar
...

--- top_k=5 (6611 chars of context) ---
Uloga bana Josipa Jelačića u revoluciji 1848. bila je višestruka:

*   **Imenovanje i politički program:** Tijekom ožujka 1848., hrvatsko konzervativno plemstvo isposlovalo je njegovo imenovanje za hrvatskog bana [Izvor: Hrvatsko Kraljevstvo (1527. – 1868.) — Revoluc

### Grounding test: out-of-domain question

What happens when the corpus **doesn't** have the answer? A well-grounded system should say so.

In [26]:
result = rag_query("Objasni osnove kvantne mehanike.")
show_rag_result("Objasni osnove kvantne mehanike.", result)
print("\n\u2192 The model should decline — this topic isn't in the corpus.")

Question: Objasni osnove kvantne mehanike.

Nemam dovoljno informacija u zadanom kontekstu.

--------------------------------------------------------------------------------
Sources:
  [-0.241] Sandro Skansi — Znanstveni i stručni tekstovi
  [-0.250] Samuel Bosch — Biografija i karijera
  [-0.261] Slaven Kosanović Lunar — Internetske stranice

→ The model should decline — this topic isn't in the corpus.


### \ud83d\udd2c Your turn

Try your own questions about Croatian history. Some ideas:
- "Tko je bio Nikola Tesla i gdje je ro\u0111en?"
- "\u0160to se dogodilo u Vukovaru 1991.?"
- "Koji su najva\u017eniji hrvatski knji\u017eevnici?"
- "Kako je nastala Dubrova\u010dka Republika?"

In [27]:
# \u2500\u2500 Try your own query \u2500\u2500
your_question = "\u0160to se dogodilo u Vukovaru 1991.?"  # \u2190 change this

result = rag_query(your_question, top_k=5)
show_rag_result(your_question, result)

Question: Što se dogodilo u Vukovaru 1991.?

Na temelju dostavljenog konteksta, u Vukovaru se 1991. godine dogodilo sljedeće:

*   **Eskalacija napetosti i početak borbi:** Napetosti u gradu počele su eskalirati već tijekom Uskrsa 1991., a situacija se dramatično pogoršala nakon pokolja dvanaest hrvatskih redarstvenika u Borovu Selu 2. svibnja 1991. godine (Izvor: Trpinjska cesta — Povijesni kontekst).
*   **Opsada i bitka:** Grad je bio pod teškim napadima jakih tenkovskih i mehaniziranih snaga Jugoslavenske narodne armije (JNA), te je ubrzo bio opkoljen i pod opsadom (Izvor: 1. mehanizirana gardijska brigada "Tigrovi" — Vukovar).
*   **Sudjelovanje postrojbi:** 
    *   U obrani grada do njegovog zauzimanja neprekidno je sudjelovala jedna bojna 3. gardijske brigade (3. gbr.) (Izvor: 3. mehanizirana gardijska brigada "Kune" — 1991.).
    *   Dijelovi 1. Gardijske Brigade ostali su unutar grada, a neki od opkoljenih vojnika uspjeli su izvesti proboj i pobjeći prije pada grada (Izvor: 1

## Key Takeaways

| Concept | What you learned |
|---|---|
| **Chunking** | Right size and strategy matter more than the LLM you pick |
| **Section-based chunking** | Respects document structure, preserves context |
| **Indexing** | Chunk \u2192 embed \u2192 store (done once) |
| **RAG query** | Retrieve \u2192 build prompt \u2192 generate (done per question) |
| **Grounding** | "Answer ONLY from context" prevents hallucination |
| **top_k tradeoff** | More chunks = more recall but more noise and cost |